# WavLM + MHFA: TidyVoice + ViMD (Run All)

Required Kaggle settings: **GPU T4 x2**, **Internet On**, and both inputs `dullahn/mozzila-tidyvoice` and `dullahn/vimd-dataset`. Use your own Kaggle account. Choose **Run All**. After completion, download `who_speak_ai_wavlm_mhfa_resource_constrained.zip` from the Output panel.

This is the predeclared compute-constrained stage: seed 42, every Train speaker, one rotating utterance per speaker per epoch, at most three epochs, patience one, full immutable Validation every epoch, and final Test once from the best Validation checkpoint.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/beaver-felix/Who-Speak-AI.git"
PINNED_REVISION = "c68471a69c089cc40a5975b22362da37abcac186"
REPOSITORY = Path("/kaggle/working/Who-Speak-AI-wavlm-worker")
PROJECT = REPOSITORY / "model/Thanh2"

if not REPOSITORY.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPOSITORY)], check=True)
elif subprocess.run(["git", "-C", str(REPOSITORY), "status", "--porcelain"], capture_output=True, text=True, check=True).stdout.strip():
    raise RuntimeError("Existing worker repository is dirty; start a fresh Kaggle session.")
subprocess.run(["git", "-C", str(REPOSITORY), "fetch", "--depth", "1", "origin", PINNED_REVISION], check=True)
subprocess.run(["git", "-C", str(REPOSITORY), "checkout", "--detach", PINNED_REVISION], check=True)
observed = subprocess.run(["git", "-C", str(REPOSITORY), "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert observed == PINNED_REVISION, (observed, PINNED_REVISION)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT}[data,wavlm_mhfa,tracking]"], check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, "Select GPU T4 x2."
for required in (Path("/kaggle/input/datasets/dullahn/mozzila-tidyvoice/TidyVoiceX_ASV"), Path("/kaggle/input/datasets/dullahn/vimd-dataset")):
    assert required.is_dir(), f"Attach the required Kaggle dataset: {required}"
print("PINNED WAVLM+MHFA WORKER READY", observed)

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "scripts/run_resource_constrained_worker.py"), "--model", "wavlm_mhfa"], cwd=PROJECT, check=True)
archive = Path("/kaggle/working/who_speak_ai_wavlm_mhfa_resource_constrained.zip")
assert archive.is_file() and archive.stat().st_size > 0
print(f"COMPLETE - download from Kaggle Output panel: {archive}")